# Load Dataset

In [1]:
import pandas as pd

orders   = pd.read_csv("olist_orders_dataset.csv")
reviews  = pd.read_csv("olist_order_reviews_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")

# Cek setiap tabel
print(orders.shape)
print(reviews.shape)
print(payments.shape)
print(customers.shape)

(99441, 8)
(99224, 7)
(103886, 5)
(99441, 5)


# Inspeksi setiap tabel

In [2]:
# Lihat kolom dan tipe data semua tabel
for name, df in [("orders", orders), ("reviews", reviews),
                  ("payments", payments), ("customers", customers)]:
    print(f"\n{'='*45}")
    print(f"TABEL: {name}")
    print(df.dtypes)
    print(f"Missing values:\n{df.isnull().sum()}")


TABEL: orders
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

TABEL: reviews
review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object
Missing values:
review_id                      0
o

# Data Cleaning

In [3]:
# Konversi semua kolom tanggal di tabel orders
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Filter hanya order yang sudah delivered
orders_delivered = orders[orders["order_status"] == "delivered"].copy()

# Hitung keterlambatan (positif = terlambat, negatif = lebih cepat)
orders_delivered["delivery_delay"] = (
    orders_delivered["order_delivered_customer_date"] -
    orders_delivered["order_estimated_delivery_date"]
).dt.days

print(f"Total order delivered: {len(orders_delivered):,}")
print(f"\nStatistik keterlambatan (hari):")
print(orders_delivered["delivery_delay"].describe().round(2))

Total order delivered: 96,478

Statistik keterlambatan (hari):
count    96470.00
mean       -11.88
std         10.18
min       -147.00
25%        -17.00
50%        -12.00
75%         -7.00
max        188.00
Name: delivery_delay, dtype: float64


### cek berapa yang terlambat vs tepat waktu

In [4]:
# Kategorikan status pengiriman
orders_delivered["delivery_status"] = orders_delivered["delivery_delay"].apply(
    lambda x: "Terlambat" if x > 0 else "Tepat Waktu / Lebih Cepat"
)

# Proporsi
status_counts = orders_delivered["delivery_status"].value_counts()
status_pct = orders_delivered["delivery_status"].value_counts(normalize=True) * 100

print("Jumlah order:")
print(status_counts)
print("\nProporsi:")
print(status_pct.round(2))

Jumlah order:
delivery_status
Tepat Waktu / Lebih Cepat    89944
Terlambat                     6534
Name: count, dtype: int64

Proporsi:
delivery_status
Tepat Waktu / Lebih Cepat    93.23
Terlambat                     6.77
Name: proportion, dtype: float64


**Fakta**: 93,23% order Olist terkirim tepat waktu atau lebih cepat dari estimasi, dengan rata-rata 12 hari lebih cepat. Namun 6,77% order mengalami keterlambatan dengan maksimum 188 hari.

**Kenapa**: Olist kemungkinan menetapkan estimasi pengiriman yang konservatif sebagai strategi manajemen ekspektasi pelanggan. Keterlambatan ekstrem hingga 188 hari mengindikasikan adanya kasus operasional yang tidak tertangani dengan baik.

**Rekomendasi**: Investigasi order terlambat berdasarkan seller dan wilayah untuk mengidentifikasi akar masalah, serta tetapkan SLA (Service Level Agreement) maksimum keterlambatan yang dapat ditoleransi.

### cek missing value

In [9]:
# Cek missing values semua tabel
for name, df in [("orders", orders), ("reviews", reviews),
                  ("payments", payments), ("customers", customers)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"\n{'='*45}")
    print(f"TABEL: {name}")
    if missing.empty:
        print("Tidak ada missing values ✅")
    else:
        print(missing)


TABEL: orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

TABEL: reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

TABEL: payments
Tidak ada missing values ✅

TABEL: customers
Tidak ada missing values ✅


**Tabel orders**:
Kolom tanggal yang missing tidak perlu diisi, karena kita sudah filter hanya order delivered. Order yang belum delivered memang wajar tidak punya tanggal pengiriman. Jadi tidak perlu handling khusus, filter delivered sudah cukup.

**Tabel reviews**:
review_comment_title dan review_comment_message missing 88% dan 59%, ini normal karena komentar tidak wajib diisi pelanggan. Kolom ini tidak akan dipakai di analisis kita, jadi biarkan saja.

**Kesimpulan**: tidak ada handling missing values yang perlu dilakukan

###  Join tabel

In [10]:
# Gabungkan semua tabel yang dibutuhkan
df_final = orders_delivered[["order_id", "customer_id",
                              "delivery_delay", "delivery_status"]].merge(
    reviews[["order_id", "review_score"]], on="order_id", how="left"
).merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left"
).merge(
    payments.groupby("order_id")["payment_type"].first().reset_index(),
    on="order_id", how="left"
)

print(f"Shape final: {df_final.shape}")
print(f"Missing values:\n{df_final.isnull().sum()}")
print(df_final.head())

Shape final: (97007, 7)
Missing values:
order_id             0
customer_id          0
delivery_delay       8
delivery_status      0
review_score       646
customer_state       0
payment_type         1
dtype: int64
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

   delivery_delay            delivery_status  review_score customer_state  \
0            -8.0  Tepat Waktu / Lebih Cepat           4.0             SP   
1            -6.0  Tepat Waktu / Lebih Cepat           4.0             BA   
2           -18.0  Tepat Waktu / Lebih Cepat           5.0             GO   
3           -13.0  Tepat Waktu / Lebih 

1. delivery_delay: 8 (sangat sedikit, drop saja)
2. review_score: 646 (order tanpa review, drop saja)  
3. payment_type: 1 (drop saja)

In [11]:
# Drop baris dengan missing values
df_final = df_final.dropna()

print(f"Shape setelah drop: {df_final.shape}")
print(f"Missing values:\n{df_final.isnull().sum()}")

Shape setelah drop: (96352, 7)
Missing values:
order_id           0
customer_id        0
delivery_delay     0
delivery_status    0
review_score       0
customer_state     0
payment_type       0
dtype: int64


# Data final

In [13]:
# Export ke CSV
df_final.to_csv("olist_cleaned.csv", index=False)
print("✅ Data bersih tersimpan: olist_cleaned.csv")

✅ Data bersih tersimpan: olist_cleaned.csv


# EDA


**Customer Experience**

"Seberapa puas pelanggan dan faktor apa yang mempengaruhinya?"

Fokus: review score, keterlambatan pengiriman, metode pembayaran

Pertanyaan bisnis:

1. Berapa rata-rata review score keseluruhan?
2. Apakah order yang terlambat mendapat review score lebih rendah?
3. Metode pembayaran apa yang paling sering digunakan?
4. State mana yang paling banyak komplain (review score rendah)?




In [14]:
# EDA menggunakan data final yang sudah bersih
print("PERTANYAAN 1 — Rata-rata Review Score")
print(f"Rata-rata: {df_final['review_score'].mean():.2f}")
print(df_final['review_score'].value_counts().sort_index())

print("\nPERTANYAAN 2 — Keterlambatan vs Review Score")
print(df_final.groupby("delivery_status")["review_score"].mean().round(2))

print("\nPERTANYAAN 3 — Metode Pembayaran")
print(df_final["payment_type"].value_counts())

print("\nPERTANYAAN 4 — State dengan Review Rendah")
state = df_final.groupby("customer_state")["review_score"].agg(["mean","count"]).round(2)
state.columns = ["Avg Score", "Jumlah"]
print(state[state["Jumlah"] >= 100].sort_values("Avg Score").head(10))

PERTANYAAN 1 — Rata-rata Review Score
Rata-rata: 4.16
review_score
1.0     9404
2.0     2941
3.0     7961
4.0    18987
5.0    57059
Name: count, dtype: int64

PERTANYAAN 2 — Keterlambatan vs Review Score
delivery_status
Tepat Waktu / Lebih Cepat    4.29
Terlambat                    2.27
Name: review_score, dtype: float64

PERTANYAAN 3 — Metode Pembayaran
payment_type
credit_card    73110
boleto         19179
voucher         2579
debit_card      1484
Name: count, dtype: int64

PERTANYAAN 4 — State dengan Review Rendah
                Avg Score  Jumlah
customer_state                   
AL                   3.84     398
MA                   3.84     716
PA                   3.91     939
SE                   3.91     334
BA                   3.93    3246
CE                   3.94    1276
RJ                   3.96   12281
PI                   4.00     472
ES                   4.08    1978
PB                   4.08     513


**Fakta**: Rata-rata review score 4.16 dari 5. Order yang tepat waktu mendapat score rata-rata 4.29, sedangkan order terlambat hanya 2.27 dengan selisih 2 poin. State RJ memiliki order terbanyak (12.281) namun review score-nya rendah (3.96).

**Kenapa**: Rendahnya review score pada order terlambat langsung terbukti dari data yaitu selisih 2 poin antara tepat waktu vs terlambat menunjukkan keterlambatan adalah faktor utama ketidakpuasan pelanggan.

**Rekomendasi**: Prioritaskan perbaikan pengiriman di state RJ karena kombinasi jumlah order besar dan review score rendah memberikan dampak bisnis terbesar. Tetapkan SLA maksimum keterlambatan untuk menjaga review score di atas 4.